In [6]:
# ==============================================================================
# SCRIPT: Train a Single DeBERTa Model for Sentiment Analysis
# GOAL: Use the specified training and test datasets correctly to build and
#       evaluate a robust model without data leakage.
# VERSION: 7.0 (Corrected file usage as per clarification)
# ==============================================================================

import os
import shutil
import joblib
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight
import gc

# ==============================================================================
# CONFIGURATION
# ==============================================================================
CONFIG = {
    "model_name": 'microsoft/deberta-v3-base',
    "max_length": 128,
    "validation_set_size": 0.2, # 20% of the training data will be used for validation
    "batch_size": 16,
    "learning_rate": 2e-5,
    "num_epochs": 5,
    "weight_decay": 0.01,
    "early_stopping_patience": 2,
    "focal_loss_gamma": 2.0,
    "output_dir": './sentiment_deberta_model/',
    "final_zip_name": 'sentiment_deberta_model',
    "random_state": 42
}

# ==============================================================================
# HELPER CLASSES & FUNCTIONS
# ==============================================================================
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean()

def standardize_and_clean_labels(df, label_col='label'):
    df.dropna(subset=[label_col], inplace=True)
    LABEL_MAPPING = {
        '-1': 'negative', -1: 'negative',
        '0': 'neutral',   0: 'neutral',
        '1': 'positive',  1: 'positive'
    }
    df[label_col] = df[label_col].astype(str).map(LABEL_MAPPING)
    original_len = len(df)
    df.dropna(subset=[label_col], inplace=True)
    new_len = len(df)
    if original_len > new_len:
        print(f"  WARNING: Removed {original_len - new_len} rows with unexpected/invalid labels.")
    return df

# ==============================================================================
# MAIN SCRIPT EXECUTION
# ==============================================================================

# --- 1. SETUP & DATA LOADING ---
print("--- Installing required libraries ---")
!pip install transformers[torch] scikit-learn pandas sentencepiece --quiet
print("✅ Libraries installed.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# --- CORRECTED 2-FILE WORKFLOW ---
print("\n--- Upload your TRAINING file (preprocessed_GOLDEN_dataset.csv) ---")
uploaded_train = files.upload()
df_full_train = pd.read_csv(next(iter(uploaded_train)))
print(f"✅ Training set loaded.")

print("\n--- Upload your TEST file (preprocessed_testing_dataset.csv) ---")
uploaded_test = files.upload()
df_test = pd.read_csv(next(iter(uploaded_test)))
print(f"✅ Test set loaded.")


# Clean and standardize labels for both dataframes
print("\n--- Cleaning and Standardizing Labels ---")
df_full_train = standardize_and_clean_labels(df_full_train)
df_test = standardize_and_clean_labels(df_test)

# Fit the LabelEncoder ONLY on the training data labels to prevent data leakage
le = LabelEncoder()
df_full_train['label_encoded'] = le.fit_transform(df_full_train['label'])

# --- VERIFICATION STEP ---
print("\n--- Verifying Final Test Set Composition ---")
print(df_test['label'].value_counts())
# Transform the test labels using the encoder that was fitted on the training data
df_test['label_encoded'] = le.transform(df_test['label'])
print("\n✅ Data Cleaning and Label Standardization Complete.")


# --- 2. DATA SPLITTING & PREPARATION ---
print("\n--- Splitting TRAINING data into Training and Validation sets ---")
# The validation set is now correctly split from the full training data.
# The df_test dataframe is kept separate for the final, unbiased evaluation.
df_train, df_val = train_test_split(
    df_full_train,
    test_size=CONFIG['validation_set_size'],
    random_state=CONFIG['random_state'],
    stratify=df_full_train['label_encoded']
)
print(f"  Training set size: {len(df_train)}")
print(f"  Validation set size: {len(df_val)}")
print(f"  Final Hold-out Test set size: {len(df_test)}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

train_encodings = tokenizer(df_train['cleaned_text'].astype(str).tolist(), truncation=True, padding=True, max_length=CONFIG['max_length'])
val_encodings = tokenizer(df_val['cleaned_text'].astype(str).tolist(), truncation=True, padding=True, max_length=CONFIG['max_length'])
test_encodings = tokenizer(df_test['cleaned_text'].astype(str).tolist(), truncation=True, padding=True, max_length=CONFIG['max_length'])

train_dataset = TensorDataset(torch.tensor(train_encodings['input_ids']), torch.tensor(train_encodings['attention_mask']), torch.tensor(df_train['label_encoded'].values))
val_dataset = TensorDataset(torch.tensor(val_encodings['input_ids']), torch.tensor(val_encodings['attention_mask']), torch.tensor(df_val['label_encoded'].values))
test_dataset = TensorDataset(torch.tensor(test_encodings['input_ids']), torch.tensor(test_encodings['attention_mask']), torch.tensor(df_test['label_encoded'].values))

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
print("✅ Data prepared for training.")


# --- 3. MODEL TRAINING WITH EARLY STOPPING ---
print("\n--- Initializing DeBERTa Model for Sequence Classification ---")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=len(le.classes_)
).to(device)

optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader) * CONFIG['num_epochs'])

class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label_encoded']), y=df_train['label_encoded'].values)
loss_function = FocalLoss(alpha=torch.tensor(class_weights, dtype=torch.float).to(device), gamma=CONFIG['focal_loss_gamma'])

print("\n--- Starting Model Training ---")
best_val_loss = float('inf')
patience_counter = 0
best_model_path = os.path.join(CONFIG['output_dir'], 'best_model.pt')
os.makedirs(CONFIG['output_dir'], exist_ok=True)

for epoch in range(CONFIG['num_epochs']):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_function(outputs.logits, labels)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = loss_function(outputs.logits, labels)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch + 1}/{CONFIG['num_epochs']} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Validation loss improved. Saving best model.")
    else:
        patience_counter += 1
        print(f"  -> Validation loss did not improve. Patience: {patience_counter}/{CONFIG['early_stopping_patience']}")

    if patience_counter >= CONFIG['early_stopping_patience']:
        print("  -> Early stopping triggered.")
        break

# --- 4. FINAL EVALUATION ON TEST SET ---
print("\n--- Evaluating final model on the unseen HOLDOUT Test Set ---")
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_true = []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

final_accuracy = accuracy_score(all_true, all_preds)
print(f"\nFinal Test Accuracy: {final_accuracy:.4f}\n")
print("--- Final Classification Report ---")
print(classification_report(
    le.inverse_transform(all_true),
    le.inverse_transform(all_preds),
    target_names=le.classes_
))
print("✅ Final Model Evaluation Complete.")


# --- 5. SAVING FINAL MODEL FOR DEPLOYMENT ---
print(f"\n--- Saving the final model and assets for system implementation ---")
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
joblib.dump(le, os.path.join(CONFIG['output_dir'], 'label_encoder.joblib'))

print(f"✅ All model components saved to '{CONFIG['output_dir']}'.")

shutil.make_archive(CONFIG['final_zip_name'], 'zip', CONFIG['output_dir'])
files.download(f"{CONFIG['final_zip_name']}.zip")

print("\n✅ Model Saving and Download Complete.")
print("*** ENTIRE SCRIPT FINISHED ***")

--- Installing required libraries ---
✅ Libraries installed.
✅ Using device: cuda

--- Upload your TRAINING file (preprocessed_GOLDEN_dataset.csv) ---


Saving preprocessed_GOLDEN_dataset.csv to preprocessed_GOLDEN_dataset (4).csv
✅ Training set loaded.

--- Upload your TEST file (preprocessed_testing_dataset.csv) ---


Saving preprocessed_testing_dataset.csv to preprocessed_testing_dataset (3).csv
✅ Test set loaded.

--- Cleaning and Standardizing Labels ---

--- Verifying Final Test Set Composition ---
label
neutral     57
negative    54
positive    50
Name: count, dtype: int64

✅ Data Cleaning and Label Standardization Complete.

--- Splitting TRAINING data into Training and Validation sets ---
  Training set size: 8297
  Validation set size: 2075
  Final Hold-out Test set size: 161


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✅ Data prepared for training.

--- Initializing DeBERTa Model for Sequence Classification ---


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Starting Model Training ---
Epoch 1/5 | Train Loss: 0.0663 | Val Loss: 0.0066
  -> Validation loss improved. Saving best model.
Epoch 2/5 | Train Loss: 0.0023 | Val Loss: 0.0084
  -> Validation loss did not improve. Patience: 1/2
Epoch 3/5 | Train Loss: 0.0003 | Val Loss: 0.0063
  -> Validation loss improved. Saving best model.
Epoch 4/5 | Train Loss: 0.0003 | Val Loss: 0.0064
  -> Validation loss did not improve. Patience: 1/2
Epoch 5/5 | Train Loss: 0.0001 | Val Loss: 0.0065
  -> Validation loss did not improve. Patience: 2/2
  -> Early stopping triggered.

--- Evaluating final model on the unseen HOLDOUT Test Set ---

Final Test Accuracy: 0.9565

--- Final Classification Report ---
              precision    recall  f1-score   support

    negative       0.98      0.89      0.93        54
     neutral       0.92      0.98      0.95        57
    positive       0.98      1.00      0.99        50

    accuracy                           0.96       161
   macro avg       0.96      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Model Saving and Download Complete.
*** ENTIRE SCRIPT FINISHED ***
